# Monte Carlo Heston Calibration


In [2]:
from __future__ import annotations

import csv
import json
import time
from pathlib import Path

import numpy as np
import pandas as pd
from scipy.optimize import minimize
from scipy.stats import norm

PARAMETER_COLUMNS = ["v0", "kappa", "theta", "sigma_v", "rho"]


## Settings


In [3]:
MARKET_DATA_PATH = Path("data/spy_options_clean.csv")
TRAINING_DATA_PATH = Path("data/simulated_training_data.csv")
OUTPUT_DIR = Path("outputs/monte_carlo_calibration")

RISK_FREE_RATE = 0.03
MAXITER = 60
FTOL = 1e-8

N_PATHS = 20000
N_STEPS_PER_YEAR = 252
RANDOM_SEED = 12345

# Use "all", "call", or "put".
OPTION_TYPE = "all"

MIN_MARKET_IV = 0.0
MAX_MARKET_IV = 5.0


## Data Helpers


In [4]:
def load_market_data(path: Path) -> pd.DataFrame:
    # Read market IVs and convert expiry dates into year fractions.
    required = {"strike", "expiry", "pull_date", "impliedVolatility", "S0"}
    market = pd.read_csv(path)
    missing = sorted(required - set(market.columns))
    if missing:
        raise ValueError(f"{path} is missing required columns: {missing}")

    expiry = pd.to_datetime(market["expiry"], errors="coerce")
    pull_date = pd.to_datetime(market["pull_date"], errors="coerce")
    market = market.assign(
        K=pd.to_numeric(market["strike"], errors="coerce"),
        S0=pd.to_numeric(market["S0"], errors="coerce"),
        T=(expiry - pull_date).dt.days / 365.0,
        market_iv=pd.to_numeric(market["impliedVolatility"], errors="coerce"),
    )

    if "dividend_yield" not in market.columns:
        market["dividend_yield"] = 0.0
    if "type" not in market.columns:
        market["type"] = "call"

    market["dividend_yield"] = pd.to_numeric(market["dividend_yield"], errors="coerce").fillna(0.0)
    market["type"] = market["type"].astype(str).str.lower()

    finite = np.isfinite(market[["K", "S0", "T", "market_iv", "dividend_yield"]]).all(axis=1)
    positive = (market["S0"] > 0.0) & (market["T"] > 0.0)
    iv_range = market["market_iv"].between(MIN_MARKET_IV, MAX_MARKET_IV)
    known_type = market["type"].isin(["call", "put"])
    return market.loc[finite & positive & iv_range & known_type].copy()


def parameter_bounds(training_data: Path) -> list[tuple[float, float]]:
    # Use the same parameter range that generated the synthetic Heston surfaces.
    training = pd.read_csv(training_data, usecols=PARAMETER_COLUMNS)
    return [(float(training[column].min()), float(training[column].max())) for column in PARAMETER_COLUMNS]


def initial_guess(bounds: list[tuple[float, float]]) -> np.ndarray:
    return np.array([(low + high) / 2.0 for low, high in bounds], dtype=float)


def make_contract_groups(market: pd.DataFrame) -> list[dict[str, object]]:
    groups = []
    for _, group in market.groupby(["S0", "T", "dividend_yield"], sort=False):
        groups.append({
            "positions": group.index.to_numpy(),
            "strikes": group["K"].to_numpy(dtype=float),
            "market_iv": group["market_iv"].to_numpy(dtype=float),
            "option_type": group["type"].to_numpy(dtype=str),
            "S0": float(group["S0"].iloc[0]),
            "T": float(group["T"].iloc[0]),
            "q": float(group["dividend_yield"].iloc[0]),
        })
    return groups


## Monte Carlo and IV Helpers


In [5]:
def params_to_dict(values: np.ndarray) -> dict[str, float]:
    return {column: float(value) for column, value in zip(PARAMETER_COLUMNS, values)}


def make_random_numbers(groups: list[dict[str, object]], seed: int) -> list[dict[str, np.ndarray]]:
    # Common random numbers keep the objective deterministic across parameter guesses.
    rng = np.random.default_rng(seed)
    random_numbers = []
    for group in groups:
        n_steps = max(1, int(np.ceil(group["T"] * N_STEPS_PER_YEAR)))
        random_numbers.append({
            "z1": rng.standard_normal((n_steps, N_PATHS)),
            "z2": rng.standard_normal((n_steps, N_PATHS)),
        })
    return random_numbers


def simulate_terminal_prices(S0: float, T: float, r: float, q: float, params: dict[str, float], z1: np.ndarray, z2: np.ndarray) -> np.ndarray:
    n_steps = z1.shape[0]
    dt = T / n_steps
    sqrt_dt = np.sqrt(dt)

    log_s = np.full(z1.shape[1], np.log(S0), dtype=float)
    v = np.full(z1.shape[1], params["v0"], dtype=float)
    rho = np.clip(params["rho"], -0.999, 0.999)
    sqrt_one_minus_rho2 = np.sqrt(1.0 - rho * rho)

    for step in range(n_steps):
        v_pos = np.maximum(v, 0.0)
        dw_v = z1[step]
        dw_s = rho * z1[step] + sqrt_one_minus_rho2 * z2[step]
        log_s += (r - q - 0.5 * v_pos) * dt + np.sqrt(v_pos) * sqrt_dt * dw_s
        v += params["kappa"] * (params["theta"] - v_pos) * dt + params["sigma_v"] * np.sqrt(v_pos) * sqrt_dt * dw_v
        v = np.maximum(v, 0.0)

    return np.exp(log_s)


def monte_carlo_prices(st: np.ndarray, strikes: np.ndarray, option_type: np.ndarray, r: float, T: float) -> np.ndarray:
    call_payoffs = np.maximum(st[:, None] - strikes[None, :], 0.0)
    put_payoffs = np.maximum(strikes[None, :] - st[:, None], 0.0)
    payoffs = np.where(option_type[None, :] == "call", call_payoffs, put_payoffs)
    return np.exp(-r * T) * payoffs.mean(axis=0)


def black_scholes_price(S0, strikes, T, r, q, sigma, option_type):
    vol_sqrt_t = np.maximum(sigma, 1e-12) * np.sqrt(T)
    d1 = (np.log(S0 / strikes) + (r - q + 0.5 * sigma**2) * T) / vol_sqrt_t
    d2 = d1 - vol_sqrt_t
    call = S0 * np.exp(-q * T) * norm.cdf(d1) - strikes * np.exp(-r * T) * norm.cdf(d2)
    put = strikes * np.exp(-r * T) * norm.cdf(-d2) - S0 * np.exp(-q * T) * norm.cdf(-d1)
    return np.where(option_type == "call", call, put)


def implied_volatility_from_prices(prices, S0, strikes, T, r, q, option_type, initial_iv):
    prices = np.asarray(prices, dtype=float)
    strikes = np.asarray(strikes, dtype=float)
    option_type = np.asarray(option_type, dtype=str)
    sigma = np.clip(np.asarray(initial_iv, dtype=float), 1e-4, 5.0)

    call_lower = np.maximum(S0 * np.exp(-q * T) - strikes * np.exp(-r * T), 0.0)
    put_lower = np.maximum(strikes * np.exp(-r * T) - S0 * np.exp(-q * T), 0.0)
    lower = np.where(option_type == "call", call_lower, put_lower)
    upper = np.where(option_type == "call", S0 * np.exp(-q * T), strikes * np.exp(-r * T))
    prices = np.clip(np.nan_to_num(prices, nan=lower), lower + 1e-8, upper - 1e-8)

    for _ in range(12):
        vol_sqrt_t = sigma * np.sqrt(T)
        d1 = (np.log(S0 / strikes) + (r - q + 0.5 * sigma**2) * T) / vol_sqrt_t
        vega = S0 * np.exp(-q * T) * norm.pdf(d1) * np.sqrt(T)
        bs_price = black_scholes_price(S0, strikes, T, r, q, sigma, option_type)
        step = np.where(vega > 1e-10, (bs_price - prices) / vega, 0.0)
        sigma = np.clip(sigma - step, 1e-4, 5.0)

    return sigma


## Calibration Helpers


In [6]:
def predict_market_iv(
    params: dict[str, float],
    groups: list[dict[str, object]],
    random_numbers: list[dict[str, np.ndarray]],
    n_contracts: int,
) -> np.ndarray:
    predicted_iv = np.full(n_contracts, np.nan, dtype=float)

    for group, draws in zip(groups, random_numbers):
        st = simulate_terminal_prices(
            group["S0"],
            group["T"],
            RISK_FREE_RATE,
            group["q"],
            params,
            draws["z1"],
            draws["z2"],
        )
        prices = monte_carlo_prices(st, group["strikes"], group["option_type"], RISK_FREE_RATE, group["T"])
        predicted_iv[group["positions"]] = implied_volatility_from_prices(
            prices,
            group["S0"],
            group["strikes"],
            group["T"],
            RISK_FREE_RATE,
            group["q"],
            group["option_type"],
            group["market_iv"],
        )

    return predicted_iv


def squared_iv_error(predicted_iv: np.ndarray, market_iv: np.ndarray) -> float:
    valid = np.isfinite(predicted_iv)
    residual = predicted_iv[valid] - market_iv[valid]
    invalid_penalty = 100.0 * np.sum(~valid)
    return float(np.sum(residual * residual) + invalid_penalty)


## Run Calibration


In [7]:
# Load and optionally filter market contracts.
market = load_market_data(MARKET_DATA_PATH)
if OPTION_TYPE != "all" and "type" in market.columns:
    market = market.loc[market["type"].str.lower() == OPTION_TYPE].copy()

if market.empty:
    raise ValueError("No market contracts remain after filtering.")

market = market.reset_index(drop=True)
market_iv = market["market_iv"].to_numpy(dtype=float)
contract_groups = make_contract_groups(market)
random_numbers = make_random_numbers(contract_groups, RANDOM_SEED)
bounds = parameter_bounds(TRAINING_DATA_PATH)


def objective(values: np.ndarray) -> float:
    predicted_iv = predict_market_iv(params_to_dict(values), contract_groups, random_numbers, len(market))
    return squared_iv_error(predicted_iv, market_iv)


started_at = time.perf_counter()
result = minimize(
    objective,
    x0=initial_guess(bounds),
    method="L-BFGS-B",
    bounds=bounds,
    options={"maxiter": MAXITER, "ftol": FTOL},
)
wall_clock_seconds = time.perf_counter() - started_at

fitted_params = np.asarray(result.x, dtype=float)
predicted_iv = predict_market_iv(params_to_dict(fitted_params), contract_groups, random_numbers, len(market))
residual = predicted_iv - market_iv
squared_error = residual * residual
valid_iv = np.isfinite(predicted_iv)
total_error = float(np.nansum(squared_error))

summary = {
    "success": bool(result.success),
    "message": str(result.message),
    "contracts": int(len(market)),
    "valid_model_ivs": int(valid_iv.sum()),
    "evaluations": int(result.nfev),
    "fitted_parameters": params_to_dict(fitted_params),
    "total_calibration_error": total_error,
    "rmse": float(np.sqrt(np.nanmean(squared_error))),
    "mae": float(np.nanmean(np.abs(residual))),
    "wall_clock_seconds": float(wall_clock_seconds),
    "bounds": {column: list(bound) for column, bound in zip(PARAMETER_COLUMNS, bounds)},
    "market_data": str(MARKET_DATA_PATH),
    "risk_free_rate": RISK_FREE_RATE,
    "option_type": OPTION_TYPE,
    "mc_settings": {"n_paths": N_PATHS, "steps_per_year": N_STEPS_PER_YEAR, "random_seed": RANDOM_SEED},
}

summary


/var/folders/j8/l8v_mngj6kgcf3q2hjtpw69w0000gn/T/ipykernel_1676/1000895328.py:72: RuntimeWarning: divide by zero encountered in divide
  step = np.where(vega > 1e-10, (bs_price - prices) / vega, 0.0)


{'success': True,
 'message': 'CONVERGENCE: RELATIVE REDUCTION OF F <= FACTR*EPSMCH',
 'contracts': 973,
 'valid_model_ivs': 973,
 'evaluations': 144,
 'fitted_parameters': {'v0': 0.02244440119517437,
  'kappa': 2.5416923057815413,
  'theta': 0.010887162570434964,
  'sigma_v': 0.9993863554271026,
  'rho': -0.10744621267088876},
 'total_calibration_error': 8.649392473833323,
 'rmse': 0.09428364888954692,
 'mae': 0.06762767233399626,
 'wall_clock_seconds': 16.464334624935873,
 'bounds': {'v0': [0.0100207958590628, 0.2498920421524241],
  'kappa': [0.1001058627817415, 4.998386022881946],
  'theta': [0.0101229951136501, 0.2499243545643585],
  'sigma_v': [0.0501999396475361, 0.9993863554271026],
  'rho': [-0.9495768662885722, -0.0005757410491344]},
 'market_data': 'data/spy_options_clean.csv',
 'risk_free_rate': 0.03,
 'option_type': 'all',
 'mc_settings': {'n_paths': 20000,
  'steps_per_year': 252,
  'random_seed': 12345}}

## Save Results


In [8]:
# Save contract-level model IVs and residuals.
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

predictions = market.copy()
for column, value in zip(PARAMETER_COLUMNS, fitted_params):
    predictions[f"fitted_{column}"] = value

predictions["mc_predicted_iv"] = predicted_iv
predictions["iv_residual"] = residual
predictions["iv_squared_error"] = squared_error
predictions.to_csv(OUTPUT_DIR / "calibration_predictions.csv", index=False)

with (OUTPUT_DIR / "calibration_summary.json").open("w") as f:
    json.dump(summary, f, indent=2)
    f.write("\n")

with (OUTPUT_DIR / "calibration_summary.csv").open("w", newline="") as f:
    writer = csv.writer(f)
    writer.writerow([
        *PARAMETER_COLUMNS,
        "total_calibration_error",
        "rmse",
        "mae",
        "wall_clock_seconds",
        "contracts",
        "valid_model_ivs",
        "evaluations",
        "success",
        "message",
    ])
    writer.writerow([
        *[summary["fitted_parameters"][column] for column in PARAMETER_COLUMNS],
        summary["total_calibration_error"],
        summary["rmse"],
        summary["mae"],
        summary["wall_clock_seconds"],
        summary["contracts"],
        summary["valid_model_ivs"],
        summary["evaluations"],
        summary["success"],
        summary["message"],
    ])

print(f"Saved calibration summary to {OUTPUT_DIR / 'calibration_summary.csv'}")
print(f"Saved per-contract predictions to {OUTPUT_DIR / 'calibration_predictions.csv'}")


Saved calibration summary to outputs/monte_carlo_calibration/calibration_summary.csv
Saved per-contract predictions to outputs/monte_carlo_calibration/calibration_predictions.csv
